# Analyse Exploratoire des Données (EDA)

## Objectif
Ce notebook a pour objectif d’explorer les données commerciales et financières de l’entreprise afin de :

- comprendre la structure et la qualité des données,
- analyser les ventes, les dépenses et les budgets,
- détecter les anomalies éventuelles,
- identifier des insights utiles à la prise de décision.

## Tables utilisées
- **Client** : informations sur les clients
- **Produit** : informations sur les produits
- **Vente** : transactions commerciales
- **Dépense** : dépenses de l’entreprise
- **Budget** : budgets alloués
- **Catégorie** : classification comptable / départementale

## Questions métier
- Quel est le chiffre d’affaires global et son évolution dans le temps ?
- Quels segments, pays, clients et produits contribuent le plus au CA ?
- Comment évoluent les dépenses ?
- Les dépenses réelles respectent-elles le budget ?
- Existe-t-il des anomalies, incohérences ou relations statistiques significatives ?

## 1. Importation des bibliothèques
Cette section charge les bibliothèques nécessaires à l’analyse, la visualisation et aux tests statistiques.

In [1]:
#importer les bibliothèques nécessaires.
import pandas as pd
import numpy as np

## 2. Chargement des données
On charge les différentes tables nécessaires à l’analyse.

In [2]:
#Cette cellule charge les différents fichiers CSV du projet dans des DataFrames pandas.

df_client=pd.read_csv('Client.csv')
df_produit=pd.read_csv('Produit.csv')
df_vente=pd.read_csv('Vente&Facture.csv')
df_depence=pd.read_csv('Dépence & Charge.csv')
df_budget=pd.read_csv('Budget_Mensuelles.csv')
df_categorie=pd.read_csv('Mapping catégories → comptes.csv')

In [3]:
# Cette cellule affiche les dimensions de chaque table
# pour avoir une première vue du volume de données.

dfs = {
    "client": df_client,
    "produit": df_produit,
    "vente": df_vente,
    "depence": df_depence,
    "budget": df_budget,
    "categorie": df_categorie
}

for name, df in dfs.items():
    print(f"{name:<10} : {df.shape[0]} lignes x {df.shape[1]} colonnes")

client     : 120 lignes x 5 colonnes
produit    : 35 lignes x 4 colonnes
vente      : 26121 lignes x 10 colonnes
depence    : 4852 lignes x 8 colonnes
budget     : 840 lignes x 4 colonnes
categorie  : 10 lignes x 4 colonnes


## Interprétation
Cette étape permet de vérifier que les tables ont bien été chargées et de comparer leur poids relatif.

À commenter dans le rapport :
- les tables les plus volumineuses sont généralement les tables transactionnelles (ventes, dépenses) ;
- les tables clients, produits et mapping sont des tables de référence ;
- un volume anormalement faible ou élevé peut révéler un problème d’import.

In [4]:
# Cette cellule affiche les premières lignes de chaque table
# pour comprendre rapidement leur structure.

for name, df in dfs.items():
    print(f"\n la table : {name}")
    display(df.head())


 la table : client


,customer_id,customer_name,country_iso3,segment,created_date
0,C0001,Client 1,USA,SMB,2018-03-02
1,C0002,Client 2,FRA,SMB,2024-02-15
2,C0003,Client 3,DZA,Retail,2017-10-03
3,C0004,Client 4,USA,Retail,2023-10-07
4,C0005,Client 5,MAR,Retail,2020-02-04



 la table : produit


,product_id,product_name,category,unit_price_eur
0,P001,Produit 1,Training,508.53
1,P002,Produit 2,Subscription,91.99
2,P003,Produit 3,Service,422.80
3,P004,Produit 4,Service,265.70
4,P005,Produit 5,Training,609.82



 la table : vente


...

,expense_id,date,category,department,vendor,amount_mad,currency,type
0,EXP000000001,2019-01-01,Rent,Facilities,Office landlord,11892.93,MAD,Fixed
1,EXP000000002,2019-01-01,Software,IT,Microsoft,4235.23,MAD,Fixed
2,EXP000000003,2019-01-01,Travel,Operations,Meta,1307.46,MAD,Variable
3,EXP000000004,2019-01-02,ProfessionalServices,Admin,Local supplier,1060.42,MAD,Variable
4,EXP000000005,2019-01-02,Logistics,Operations,Tax authority,692.28,MAD,Variable



 la table : budget


,month,category,department,budget_amount_mad
0,2019-01-01,Salaries,RH,43637.41
1,2019-01-01,Rent,Facilities,12110.32
2,2019-01-01,Utilities,Facilities,1520.34
3,2019-01-01,Marketing,Growth,7956.21
4,2019-01-01,Software,IT,6457.25



 la table : categorie


,category,department,account_code,account_label
0,Salaries,RH,6000,Compte Salaries
1,Rent,Facilities,6001,Compte Rent
2,Utilities,Facilities,6002,Compte Utilities
3,Marketing,Growth,6003,Compte Marketing
4,Software,IT,6004,Compte Software


In [5]:
# Cette cellule permet d'inspecter les noms de colonnes, les types,
# ainsi que le nombre de valeurs non nulles par table.

for name, df in dfs.items():
    print(f"\nStructure : {name} ")
    display(df.info())


Structure : client 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   customer_id    120 non-null    object
 1   customer_name  120 non-null    object
 2   country_iso3   120 non-null    object
 3   segment        120 non-null    object
 4   created_date   120 non-null    object
dtypes: object(5)
memory usage: 4.8+ KB


None


Structure : produit 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35 entries, 0 to 34
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   product_id      35 non-null     object 
 1   product_name    35 non-null     object 
 2   category        35 non-null     object 
 3   unit_price_eur  35 non-null     float64
dtypes: float64(1), object(3)
memory usage: 1.2+ KB


None


Structure : vente 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26121 entries, 0 to 26120
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   date                26121 non-null  object 
 1   invoice_id          26121 non-null  object 
 2   customer_id         26121 non-null  object 
 3   product_id          26121 non-null  object 
 4   qty                 26121 non-null  int64  
 5   unit_price_eur      26121 non-null  float64
 6   discount_rate       26121 non-null  float64
 7   total_eur           26121 non-null  float64
 8   currency            26121 non-null  object 
 9   payment_terms_days  26121 non-null  int64  
dtypes: float64(3), int64(2), object(5)
memory usage: 2.0+ MB


...

None


Structure : budget 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 840 entries, 0 to 839
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   month              840 non-null    object 
 1   category           840 non-null    object 
 2   department         840 non-null    object 
 3   budget_amount_mad  840 non-null    float64
dtypes: float64(1), object(3)
memory usage: 26.4+ KB


None


Structure : categorie 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   category       10 non-null     object
 1   department     10 non-null     object
 2   account_code   10 non-null     int64 
 3   account_label  10 non-null     object
dtypes: int64(1), object(3)
memory usage: 452.0+ bytes


None

## 3. Diagnostic de qualité des données
On vérifie ici :
- les valeurs manquantes,
- les doublons,
- la structure des colonnes,
- les relations entre tables,
- les anomalies métier.

In [6]:
# Cette cellule résume, pour chaque table, le nombre de valeurs manquantes
# et le nombre de doublons complets.

quality_summary = []

for name, df in dfs.items():
    quality_summary.append({
        "table": name,
        "nb_lignes": len(df),
        "nb_colonnes": df.shape[1],
        "nb_valeurs_manquantes": int(df.isna().sum().sum()),
        "nb_doublons": int(df.duplicated().sum())
    })

quality_summary = pd.DataFrame(quality_summary)
display(quality_summary)

,table,nb_lignes,nb_colonnes,nb_valeurs_manquantes,nb_doublons
0,client,120,5,0,0
1,produit,35,4,0,0
2,vente,26121,10,0,0
3,depence,4852,8,0,0
4,budget,840,4,0,0
5,categorie,10,4,0,0


In [7]:
# Cette cellule vérifie les doublons sur des clés supposées uniques
# afin d'identifier d'éventuels problèmes d'unicité métier.

print("Doublons customer_id :", df_client["customer_id"].duplicated().sum())
print("Doublons product_id  :", df_produit["product_id"].duplicated().sum())
print("Doublons invoice_id  :", df_vente["invoice_id"].duplicated().sum())
print("Doublons expense_id  :", df_depence["expense_id"].duplicated().sum())

Doublons customer_id : 0
Doublons product_id  : 0
Doublons invoice_id  : 0
Doublons expense_id  : 0


In [8]:
# Cette cellule nettoie les espaces dans les colonnes texte et convertit les dates.

for df in [df_client, df_produit, df_vente, df_depence, df_budget, df_categorie]:
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].astype(str).str.strip()

df_vente["date"] = pd.to_datetime(df_vente["date"], errors="coerce")
df_depence["date"] = pd.to_datetime(df_depence["date"], errors="coerce")
df_budget["month"] = pd.to_datetime(df_budget["month"], errors="coerce")
df_client["created_date"] = pd.to_datetime(df_client["created_date"], errors="coerce")

## Interprétation
Ce nettoyage permet d’uniformiser les données avant analyse.

À commenter :
- la suppression des espaces évite des erreurs de jointure ou de regroupement ;
- la conversion des dates est indispensable pour l’analyse temporelle ;
- les dates non convertibles deviennent nulles et doivent être surveillées.

In [9]:
# Cette cellule vérifie la cohérence des relations ventes -> clients et ventes -> produits.

missing_customers = set(df_vente["customer_id"]) - set(df_client["customer_id"])
missing_products = set(df_vente["product_id"]) - set(df_produit["product_id"])

print("Clients manquants dans les ventes :", len(missing_customers))
print("Produits manquants dans les ventes :", len(missing_products))

Clients manquants dans les ventes : 0
Produits manquants dans les ventes : 0


## Interprétation
L’intégrité référentielle permet de vérifier que chaque vente est bien rattachée à un client et à un produit existants.

À commenter :
- si le nombre est nul, la cohérence du modèle relationnel est bonne ;
- sinon, certaines ventes ne pourront pas être enrichies par les dimensions clients ou produits ;
- cela limite la fiabilité des analyses segmentées.

In [10]:
# Cette cellule détecte les anomalies métier dans les ventes.

conditions_vente = []
conditions_vente.append(df_vente["qty"] <= 0)
conditions_vente.append(df_vente["unit_price_eur"] < 0)
conditions_vente.append(df_vente["discount_rate"] < 0)
conditions_vente.append(df_vente["discount_rate"] > 1)
conditions_vente.append(df_vente["total_eur"] < 0)

anomalie_vente = conditions_vente[0]
for cond in conditions_vente[1:]:
    anomalie_vente = anomalie_vente | cond

ventes_anormales = df_vente[anomalie_vente]

print("Nombre de ventes anormales :", len(ventes_anormales))
display(ventes_anormales.head(20))

Nombre de ventes anormales : 0


,date,invoice_id,customer_id,product_id,qty,unit_price_eur,discount_rate,total_eur,currency,payment_terms_days


## Interprétation
Cette étape identifie les enregistrements incompatibles avec les règles métier.

À commenter :
- une quantité négative ou nulle est incohérente pour une vente ;
- un prix ou un total négatif est anormal ;
- un taux de remise doit rester entre 0 et 1 ;
- ces lignes doivent être corrigées ou exclues selon la politique de nettoyage.

In [11]:
# Cette cellule rattache les dépenses et les budgets au plan comptable.

for df in [df_depence, df_budget, df_categorie]:
    df["category"] = df["category"].astype(str).str.strip()
    df["department"] = df["department"].astype(str).str.strip()

df_depence_clean = df_depence.merge(
    df_categorie[["category", "department", "account_code", "account_label"]],
    on=["category", "department"],
    how="left"
)

df_budget_clean = df_budget.merge(
    df_categorie[["category", "department", "account_code", "account_label"]],
    on=["category", "department"],
    how="left"
)

print("Dépenses sans mapping :", df_depence_clean["account_code"].isna().sum())
print("Budgets sans mapping :", df_budget_clean["account_code"].isna().sum())

Dépenses sans mapping : 0
Budgets sans mapping : 0


## Interprétation
Le mapping comptable permet de relier les charges et budgets à des comptes analytiques.

À commenter :
- un mapping complet permet des analyses financières plus fines ;
- les lignes non mappées empêchent une lecture correcte par compte ;
- ces exceptions doivent être corrigées avant la phase KPI.

In [13]:
import os

# Créer le dossier automatiquement
os.makedirs("Data/processed", exist_ok=True)

In [14]:
# Cette cellule sauvegarde les données nettoyées.

df_client.to_csv("Data/processed/clean_client.csv", index=False)
df_produit.to_csv("Data/processed/clean_produit.csv", index=False)
df_vente.to_csv("Data/processed/clean_vente.csv", index=False)
df_depence_clean.to_csv("Data/processed/clean_depence.csv", index=False)
df_budget_clean.to_csv("Data/processed/clean_budget.csv", index=False)
df_categorie.to_csv("Data/processed/clean_categorie.csv", index=False)

## Conclusion
À l’issue de ce notebook :
- les données sont chargées et normalisées ;
- les principaux contrôles qualité ont été réalisés ;
- les tables nettoyées sont prêtes à être exploitées dans les notebooks statistiques et KPI.